# 01 — Exploratory Data Analysis
Connect to the existing Gold schema, inspect tables, profile the fact table and check data quality. Update field names after schema inspection.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv("../.env")
host = os.getenv("PGHOST", "localhost")
port = os.getenv("PGPORT", "5432")
database = os.getenv("PGDATABASE", "ecommerce_practice")
user = os.getenv("PGUSER", "postgres")
password = os.getenv("PGPASSWORD", "")
schema = os.getenv("PGSCHEMA", "day24_gold")

if not password:
    raise ValueError("Set PGPASSWORD in the project-root .env file.")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)
print("Connected to PostgreSQL")


## Inspect source tables

In [ ]:
tables = ["fact_sales", "dim_customer", "dim_product", "dim_region", "dim_date"]
for table in tables:
    df_preview = pd.read_sql(
        text(f'SELECT * FROM "{schema}"."{table}" LIMIT 5'), engine
    )
    print(f"\\n{table}: {df_preview.shape}")
    display(df_preview)


## Load fact data

In [ ]:
fact = pd.read_sql(text(f'SELECT * FROM "{schema}"."fact_sales"'), engine)
display(fact.head())
print("Rows:", len(fact))
display(fact.info())
display(fact.describe(include="all").T)


## Missing values and duplicates

In [ ]:
quality = pd.DataFrame({
    "missing_count": fact.isna().sum(),
    "missing_pct": (fact.isna().mean() * 100).round(2),
    "distinct_count": fact.nunique(dropna=True)
})
display(quality)
print("Exact duplicate rows:", fact.duplicated().sum())


## Numeric distributions
Select numeric columns after inspecting the schema.

In [ ]:
numeric_cols = fact.select_dtypes(include="number").columns.tolist()
display(fact[numeric_cols].hist(figsize=(14, 9), bins=25))
